<a href="https://colab.research.google.com/github/tmcol2008/agriconnect/blob/main/AGRI_CONNECT.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install fastapi uvicorn sqlalchemy psycopg2-binary passlib[bcrypt] python-jose 'pydantic[email]' python-multipart

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.3/4.3 MB 32.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 7.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 150.8/150.8 kB 5.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 525.6/525.6 kB 25.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 331.1/331.1 kB 13.2 MB/s eta 0:00:00


In [2]:
from typing import Literal
from pydantic import BaseModel, EmailStr, field_validator
from fastapi import Depends, HTTPException
from fastapi.security import OAuth2PasswordBearer
from jose import jwt, exceptions as jose_exceptions # Import jwt and exceptions

VALID_ROLES = {"buyer", "seller"}

class RegisterRequest(BaseModel):
    email: EmailStr
    password: str
    role: Literal["buyer", "seller"]   # ← rejects anything else at parse time

class ProductRequest(BaseModel):
    name: str
    description: str
    price: float
    stock: int

    @field_validator("name")
    @classmethod
    def name_not_empty(cls, v):
        if not v.strip():
            raise ValueError("name cannot be empty")
        return v.strip()

    @field_validator("price")
    @classmethod
    def price_positive(cls, v):
        if v <= 0:
            raise ValueError("price must be greater than 0")
        return round(v, 2)

    @field_validator("stock")
    @classmethod
    def stock_non_negative(cls, v):
        if v < 0:
            raise ValueError("stock cannot be negative")
        return v

class OrderRequest(BaseModel):
    product_id: str
    quantity: int

    @field_validator("quantity")
    @classmethod
    def quantity_at_least_one(cls, v):
        if v < 1:
            raise ValueError("quantity must be at least 1")
        return v

# Placeholder for OAuth2 scheme (tokenUrl should point to your token endpoint)
oauth2_scheme = OAuth2PasswordBearer(tokenUrl="token")

# Placeholder for JWT secret key and algorithm (REPLACE WITH ACTUAL SECURE VALUES)
SECRET_KEY = "your-secret-key"
ALGORITHM = "HS256"

# In get_current_user — also validate role is present:
def get_current_user(token: str = Depends(oauth2_scheme)):
    try:
        payload = jwt.decode(token, SECRET_KEY, algorithms=[ALGORITHM])
        user_id: str = payload.get("sub") # Common practice to store user_id as 'sub'
        role: str = payload.get("role")
        if not user_id or not role:
            # Use jose_exceptions.JWTError for specific JWT errors
            raise jose_exceptions.JWTError("Missing claims")
    except jose_exceptions.JWTError:
        raise HTTPException(status_code=401, detail="Invalid or expired token")
    return {"user_id": user_id, "role": role}

In [3]:
!uvicorn main:app --host 0.0.0.0 --port 8000

ERROR:    Error loading ASGI app. Could not import module "main".


In [4]:
import uvicorn
from fastapi import FastAPI, Depends, HTTPException, status
from typing import Literal
from pydantic import BaseModel, EmailStr, field_validator
from fastapi.security import OAuth2PasswordBearer
from jose import jwt, exceptions as jose_exceptions # Import jwt and exceptions

app = FastAPI()

VALID_ROLES = {"buyer", "seller"}

class RegisterRequest(BaseModel):
    email: EmailStr
    password: str
    role: Literal["buyer", "seller"]   # ← rejects anything else at parse time

class ProductRequest(BaseModel):
    name: str
    description: str
    price: float
    stock: int

    @field_validator("name")
    @classmethod
    def name_not_empty(cls, v):
        if not v.strip():
            raise ValueError("name cannot be empty")
        return v.strip()

    @field_validator("price")
    @classmethod
    def price_positive(cls, v):
        if v <= 0:
            raise ValueError("price must be greater than 0")
        return round(v, 2)

    @field_validator("stock")
    @classmethod
    def stock_non_negative(cls, v):
        if v < 0:
            raise ValueError("stock cannot be negative")
        return v

class OrderRequest(BaseModel):
    product_id: str
    quantity: int

    @field_validator("quantity")
    @classmethod
    def quantity_at_least_one(cls, v):
        if v < 1:
            raise ValueError("quantity must be at least 1")
        return v

# Placeholder for OAuth2 scheme (tokenUrl should point to your token endpoint)
oauth2_scheme = OAuth2PasswordBearer(tokenUrl="token")

# Placeholder for JWT secret key and algorithm (REPLACE WITH ACTUAL SECURE VALUES)
SECRET_KEY = "your-secret-key"
ALGORITHM = "HS256"

# In get_current_user — also validate role is present:
def get_current_user(token: str = Depends(oauth2_scheme)):
    try:
        payload = jwt.decode(token, SECRET_KEY, algorithms=[ALGORITHM])
        user_id: str = payload.get("sub") # Common practice to store user_id as 'sub'
        role: str = payload.get("role")
        if not user_id or not role:
            # Use jose_exceptions.JWTError for specific JWT errors
            raise jose_exceptions.JWTError("Missing claims")
    except jose_exceptions.JWTError:
        raise HTTPException(status_code=status.HTTP_401_UNAUTHORIZED, detail="Invalid or expired token")
    return {"user_id": user_id, "role": role}

# Define a simple root endpoint
@app.get("/", tags=["Root"])
async def read_root():
    return {"message": "Welcome to the FastAPI application!"}

# Example of a protected endpoint
@app.get("/users/me", tags=["Users"])
async def read_users_me(current_user: dict = Depends(get_current_user)):
    return current_user

app_code_content = """
from typing import Literal
from pydantic import BaseModel, EmailStr, field_validator
from fastapi import FastAPI, Depends, HTTPException, status
from fastapi.security import OAuth2PasswordBearer
from jose import jwt, exceptions as jose_exceptions

app = FastAPI()

VALID_ROLES = {"buyer", "seller"}

class RegisterRequest(BaseModel):
    email: EmailStr
    password: str
    role: Literal["buyer", "seller"]

class ProductRequest(BaseModel):
    name: str
    description: str
    price: float
    stock: int

    @field_validator("name")
    @classmethod
    def name_not_empty(cls, v):
        if not v.strip():
            raise ValueError("name cannot be empty")
        return v.strip()

    @field_validator("price")
    @classmethod
    def price_positive(cls, v):
        if v <= 0:
            raise ValueError("price must be greater than 0")
        return round(v, 2)

    @field_validator("stock")
    @classmethod
    def stock_non_negative(cls, v):
        if v < 0:
            raise ValueError("stock cannot be negative")
        return v

class OrderRequest(BaseModel):
    product_id: str
    quantity: int

    @field_validator("quantity")
    @classmethod
    def quantity_at_least_one(cls, v):
        if v < 1:
            raise ValueError("quantity must be at least 1")
        return v

oauth2_scheme = OAuth2PasswordBearer(tokenUrl="token")

SECRET_KEY = "your-secret-key"
ALGORITHM = "HS256"

def get_current_user(token: str = Depends(oauth2_scheme)):
    try:
        payload = jwt.decode(token, SECRET_KEY, algorithms=[ALGORITHM])
        user_id: str = payload.get("sub")
        role: str = payload.get("role")
        if not user_id or not role:
            raise jose_exceptions.JWTError("Missing claims")
    except jose_exceptions.JWTError:
        raise HTTPException(status_code=status.HTTP_401_UNAUTHORIZED, detail="Invalid or expired token")
    return {"user_id": user_id, "role": role}

@app.get("/", tags=["Root"])
async def read_root():
    return {"message": "Welcome to the FastAPI application!"}

@app.get("/users/me", tags=["Users"])
async def read_users_me(current_user: dict = Depends(get_current_user)):
    return current_user
"""

# Save file first
with open("main.py", "w") as f:
    f.write(app_code_content)

In [5]:
# Verify the content of main.py after writing
with open("main.py", "r") as f:
    print(f.read())


from typing import Literal
from pydantic import BaseModel, EmailStr, field_validator
from fastapi import FastAPI, Depends, HTTPException, status
from fastapi.security import OAuth2PasswordBearer
from jose import jwt, exceptions as jose_exceptions

app = FastAPI()

VALID_ROLES = {"buyer", "seller"}

class RegisterRequest(BaseModel):
    email: EmailStr
    password: str
    role: Literal["buyer", "seller"]

class ProductRequest(BaseModel):
    name: str
    description: str
    price: float
    stock: int

    @field_validator("name")
    @classmethod
    def name_not_empty(cls, v):
        if not v.strip():
            raise ValueError("name cannot be empty")
        return v.strip()

    @field_validator("price")
    @classmethod
    def price_positive(cls, v):
        if v <= 0:
            raise ValueError("price must be greater than 0")
        return round(v, 2)

    @field_validator("stock")
    @classmethod
    def stock_non_negative(cls, v):
        if v < 0:
            raise

In [6]:
!uvicorn main:app --host 0.0.0.0 --port 8080

INFO:     Started server process [20736]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
ERROR:    [Errno 98] error while attempting to bind on address ('0.0.0.0', 8080): address already in use
INFO:     Waiting for application shutdown.
INFO:     Application shutdown complete.


In [7]:
!pip install pyngrok